# 06 - Fine-Tuning (Optional)

Fine-tunes the RoBERTa sequence classification backbone (`emotion-english-distilroberta-base`) by adapting its classification head to the catalog's 4 target emotion classes (`joy`, `sadness`, `fear`, `neutral`).

---

### Workflow
- **Head Adaptation**: Replaces the original 7-class head with a new 4-class projection layer.
- **Training & Validation**: Trains on book descriptions using AdamW optimizer with stratified evaluation split.
- **Export**: Evaluates baseline vs fine-tuned accuracy and saves the tuned model and tokenizer to `models/emotion_finetuned`.


In [1]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split

from app.config import CATALOG_CSV, EMOTION_MODEL_NAME, EMOTIONS, MODELS_DIR, get_logger

logger = get_logger("fine_tuning")
RANDOM_SEED = 42

catalog = pd.read_csv(CATALOG_CSV, dtype={"isbn13": "string"})
logger.info("Loaded catalog: %d rows", len(catalog))

2026-08-17 19:50:11 | INFO    | booklens.fine_tuning | Loaded catalog: 22568 rows


## 1. Emotion Classifier Head Adaptation & Fine-Tuning

Reinitializes the output head to 4 emotion classes and fine-tunes on catalog descriptions.


In [2]:
FT_EMOTION_SIZE = 3000

emotion_df = catalog.dropna(subset=["description", "emotion"])
emotion_df = emotion_df[emotion_df["description"].str.strip() != ""]
emotion_df = emotion_df.sample(n=min(FT_EMOTION_SIZE, len(emotion_df)), random_state=RANDOM_SEED)

label_to_id = {label: i for i, label in enumerate(EMOTIONS)}
id_to_label = {i: label for i, label in enumerate(EMOTIONS)}
emotion_df = emotion_df.assign(label_id=emotion_df["emotion"].map(label_to_id))

train_df, eval_df = train_test_split(
    emotion_df, test_size=0.2, random_state=RANDOM_SEED, stratify=emotion_df["label_id"],
)
logger.info("Emotion fine-tune: %d train rows, %d eval rows", len(train_df), len(eval_df))

2026-08-17 19:50:11 | INFO    | booklens.fine_tuning | Emotion fine-tune: 2400 train rows, 600 eval rows


In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(EMOTION_MODEL_NAME)


class EmotionDataset(torch.utils.data.Dataset):
    """Tokenized (text, label) pairs for the emotion classifier fine-tune."""

    def __init__(self, texts, labels):
        self.encodings = tokenizer(list(texts), truncation=True, padding=True, max_length=256)
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(value[idx]) for key, value in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item


def collate(batch):
    keys = batch[0].keys()
    return {key: torch.stack([row[key] for row in batch]) for key in keys}


train_dataset = EmotionDataset(train_df["description"], train_df["label_id"])
eval_dataset = EmotionDataset(eval_df["description"], eval_df["label_id"])

In [4]:
from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification

emotion_model = AutoModelForSequenceClassification.from_pretrained(
    EMOTION_MODEL_NAME,
    num_labels=len(EMOTIONS),
    id2label=id_to_label,
    label2id=label_to_id,
    ignore_mismatched_sizes=True,
)
device = "mps" if torch.backends.mps.is_available() else "cpu"
emotion_model.to(device)


def evaluate_accuracy(model, dataset, batch_size=32) -> float:
    model.eval()
    loader = DataLoader(dataset, batch_size=batch_size, collate_fn=collate)
    correct, total = 0, 0
    with torch.no_grad():
        for batch in loader:
            batch = {key: value.to(device) for key, value in batch.items()}
            labels = batch.pop("labels")
            logits = model(**batch).logits
            correct += (logits.argmax(dim=1) == labels).sum().item()
            total += len(labels)
    return correct / total


baseline_emotion_accuracy = evaluate_accuracy(emotion_model, eval_dataset)
logger.info("Baseline (untrained head) accuracy: %.3f", baseline_emotion_accuracy)

optimizer = AdamW(emotion_model.parameters(), lr=2e-5)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, collate_fn=collate)

emotion_model.train()
for epoch in range(3):
    for batch in train_loader:
        batch = {key: value.to(device) for key, value in batch.items()}
        optimizer.zero_grad()
        loss = emotion_model(**batch).loss
        loss.backward()
        optimizer.step()

finetuned_emotion_accuracy = evaluate_accuracy(emotion_model, eval_dataset)
logger.info("Fine-tuned accuracy: %.3f (baseline %.3f)", finetuned_emotion_accuracy, baseline_emotion_accuracy)

[transformers] You passed `num_labels=4` which is incompatible to the `id2label` map of length `7`.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                        | Status   |                                                                                       
---------------------------+----------+---------------------------------------------------------------------------------------
classifier.out_proj.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([7, 768]) vs model:torch.Size([4, 768])
classifier.out_proj.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([7]) vs model:torch.Size([4])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


2026-08-17 19:50:22 | INFO    | booklens.fine_tuning | Baseline (untrained head) accuracy: 0.278
2026-08-17 19:54:13 | INFO    | booklens.fine_tuning | Fine-tuned accuracy: 0.858 (baseline 0.278)


In [5]:
EMOTION_FINETUNED_PATH = MODELS_DIR / "emotion_finetuned"
emotion_model.save_pretrained(str(EMOTION_FINETUNED_PATH))
tokenizer.save_pretrained(str(EMOTION_FINETUNED_PATH))
logger.info("Saved fine-tuned emotion model to %s", EMOTION_FINETUNED_PATH)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

2026-08-17 19:54:13 | INFO    | booklens.fine_tuning | Saved fine-tuned emotion model to /Users/obscure/Developer/MINOR/book_lens/models/emotion_finetuned
